In [ ]:
import rdkit.Chem as Chem
from rdkit.Chem import Descriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.environ['CONDA_PREFIX'],'share','RDKit','Contrib'))

from SA_Score import sascorer
#function below was copied from https://github.com/aksub99/MolDQN-pytorch/blob/master/utils.py
def penalized_logp(molecule):
    """Calculates the penalized logP of a molecule.

  Refactored from
  https://github.com/wengong-jin/icml18-jtnn/blob/master/bo/run_bo.py
  See Junction Tree Variational Autoencoder for Molecular Graph Generation
  https://arxiv.org/pdf/1802.04364.pdf
  Section 3.2
  Penalized logP is defined as:
   y(m) = logP(m) - SA(m) - cycle(m)
   y(m) is the penalized logP,
   logP(m) is the logP of a molecule,
   SA(m) is the synthetic accessibility score,
   cycle(m) is the largest ring size minus by six in the molecule.

  Args:
    molecule: Chem.Mol. A molecule.

  Returns:
    Float. The penalized logP value.

  """
    log_p = Descriptors.MolLogP(molecule)
    sas_score = sascorer.calculateScore(molecule)
    largest_ring_size = get_largest_ring_size(molecule)
    cycle_score = max(largest_ring_size - 6, 0)
    return log_p - sas_score - cycle_score

def get_largest_ring_size(molecule):
    """Calculates the largest ring size in the molecule.

  Refactored from
  https://github.com/wengong-jin/icml18-jtnn/blob/master/bo/run_bo.py

  Args:
    molecule: Chem.Mol. A molecule.

  Returns:
    Integer. The largest ring size.
  """
    cycle_list = molecule.GetRingInfo().AtomRings()
    if cycle_list:
        cycle_length = max([len(j) for j in cycle_list])
    else:
        cycle_length = 0
    return cycle_length


def plogp_wrapper(mols):
    """
    function that docks every mol in array mols using dockstring.
    hard filter at 30 heavy atoms to avoid model exploiting docking score.
    """
    scores = []
    for mol in mols:
        scores.append(penalized_logp(mol))
    return scores

plogp_wrapper([Chem.MolFromSmiles("CCCC"),Chem.MolFromSmiles("c1ccccc1C"),Chem.MolFromSmiles("c1ccccc1C"*6)])

In [ ]:
from lacan import lacan,gen
from rdkit.Chem import Draw
from lacan.gen import GAReporter
p = lacan.load_profile("chembl")

reporter = GAReporter(label="Penalized LogP")
winners_shape = gen.generate_optimized_molecules(
    plogp_wrapper, p,
    preset="ml",
    # scoring_budget=20,    # molecules scored per generation (preset default)
    # startN=20,            # initial population size
    generations=15,
    higher_is_better=True,
    n_jobs=12,
    quiet=False,
    seed=888,
    callback=reporter
)
allc_shape = sorted(winners_shape, key=lambda x: -x[1])
print(f"\n{len(allc_shape)} molecules in HallOfFame")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc_shape],
    useSVG=True, molsPerRow=4,
    legends=[str(round(c[1], 3)) for c in allc_shape],
    maxMols=20,
)
display(d)

In [ ]:
allc_shape = sorted(winners_shape, key=lambda x: -x[1])
print(f"\n{len(allc_shape)} molecules in HallOfFame")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc_shape],
    useSVG=True, molsPerRow=4,
    legends=[str(round(c[1], 3)) for c in allc_shape],
    maxMols=16,
)
display(d)

In [ ]:
penalized_logp(Chem.MolFromSmiles("c1ccccc1CCCCCCc1ccccc1CCCCCCCCCCCCCCCCCc1ccccc1"))

In [ ]:
reporter.plot()